# Segregation Index Decomposition

This notebook walks through the decomposition framework of the PySAL *segregation* package, which breaks the difference between two comparative segregation measures into a **spatial component** (`c_s`) and an **attribute component** (`c_a`), following *Rey, S. et al. "Comparative Spatial Segregation Analytics"*.

To keep the example self-contained we use the Sacramento demonstration dataset bundled with `libpysal` and compare segregation between the western and eastern halves of the region.

## Table of Contents
* [Data preparation](#Data-preparation)
* [Composition Approach (default)](#Composition-Approach-(default))
* [Share Approach](#Share-Approach)
* [Dual Composition Approach](#Dual-Composition-Approach)
* [Inspecting a different index: Relative Concentration](#Inspecting-a-different-index:-Relative-Concentration)

## Data preparation

First, import the needed libraries.

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from libpysal.examples import load_example
from segregation.singlegroup import Gini, RelativeConcentration
from segregation.decomposition import DecomposeSegregation

Read the Sacramento tracts and reproject them into an appropriate projected CRS so that distance-based operations behave well.

In [ ]:
sacramento = gpd.read_file(load_example("Sacramento1").get_path("sacramentot2.shp"))
sacramento = sacramento.to_crs(sacramento.estimate_utm_crs())

We study the segregation of the non-Hispanic Black population (`BLACK`) relative to the total population (`TOT_POP`). To obtain two contexts to compare, we split the region into a western and an eastern half at the median tract-centroid longitude.

In [ ]:
sacramento["cx"] = sacramento.geometry.centroid.x
split = sacramento["cx"].median()

west = sacramento.loc[sacramento["cx"] <= split].copy()
east = sacramento.loc[sacramento["cx"] > split].copy()

len(west), len(east)

The composition of the focal group in each context:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, gdf, title in zip(axes, [west, east], ["West", "East"]):
    gdf["composition"] = np.where(gdf["TOT_POP"] == 0, 0, gdf["BLACK"] / gdf["TOT_POP"])
    gdf.plot(column="composition", cmap="OrRd", legend=True, ax=ax)
    ax.set_title(f"Composition, {title}")
    ax.axis("off")

We first compare the Gini segregation index of both contexts and check the difference in point estimates.

In [ ]:
G_west = Gini(west, "BLACK", "TOT_POP")
G_east = Gini(east, "BLACK", "TOT_POP")

G_west.statistic - G_east.statistic

The options available for the decomposition are documented on the `DecomposeSegregation` class:

In [ ]:
help(DecomposeSegregation)

## Composition Approach (default)

The difference fitted above can be decomposed into a spatial component (`c_s`) and an attribute component (`c_a`). Let's estimate both.

In [ ]:
DS_composition = DecomposeSegregation(G_west, G_east)
DS_composition.c_s

In [ ]:
DS_composition.c_a

Whichever component has the larger absolute value contributes more to the observed difference. The difference in composition can be inspected with the `cdfs` plot type:

In [ ]:
DS_composition.plot(plot_type="cdfs")

When the underlying data are GeoDataFrames, the counterfactual compositions can also be mapped. The first and second contexts are West and East, respectively.

In [ ]:
DS_composition.plot(plot_type="maps")

*In every plotting method, the title reports each component of the decomposition.*

## Share Approach

The `share` approach builds the counterfactual total population of each unit from the share of both the focal and the complementary group in each context.

In [ ]:
DS_share = DecomposeSegregation(G_west, G_east, counterfactual_approach="share")
DS_share.plot(plot_type="cdfs")

In [ ]:
DS_share.plot(plot_type="maps")

## Dual Composition Approach

The `dual_composition` approach is similar to the default composition approach, but it also uses the counterfactual composition of the complementary group.

In [ ]:
DS_dual = DecomposeSegregation(G_west, G_east, counterfactual_approach="dual_composition")
DS_dual.plot(plot_type="cdfs")

In [ ]:
DS_dual.plot(plot_type="maps")

## Inspecting a different index: Relative Concentration

The decomposition works with any comparative pair of indices of the same type. Here we repeat it with the spatial `RelativeConcentration` index, where the spatial component typically plays a larger role.

In [ ]:
RCO_west = RelativeConcentration(west, "BLACK", "TOT_POP")
RCO_east = RelativeConcentration(east, "BLACK", "TOT_POP")

RCO_west.statistic - RCO_east.statistic

In [ ]:
RCO_decomp = DecomposeSegregation(RCO_west, RCO_east)
RCO_decomp.c_s

In [ ]:
RCO_decomp.c_a